# Clase 5 – Distribución Muestral
### Fundamentos de Programación Python para el Análisis de Datos

**Temas:** Distribución muestral · Ley de los Grandes Números · Teorema del Límite Central

---

## 0. Instalación y configuración de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import expon, norm

# Configuración visual
sns.set_theme(style='whitegrid')
np.random.seed(42)
print('Librerías cargadas correctamente ✅')

---
## 1. Distribución Muestral de la Media

La **distribución muestral** es la distribución de una estadística (como la media) obtenida al tomar muchas muestras aleatorias de una población.

> **Ejemplo práctico:** Si tomamos muchas muestras de 30 registros de tiempo de respuesta y calculamos la media en cada una, la distribución de esas medias forma una distribución muestral.

In [ ]:
# --- Código Slide 6: Propiedades clave ---

# Generamos una población normal (μ=50, σ=10, N=10.000)
poblacion = np.random.normal(50, 10, 10000)
media_poblacional = np.mean(poblacion)
std_poblacional   = np.std(poblacion)

print(f'Media poblacional (μ):              {media_poblacional:.2f}')
print(f'Desv. estándar poblacional (σ):     {std_poblacional:.2f}')

# Tomamos 1.000 muestras de tamaño n=30 y calculamos la media de cada una
n = 30
medias_muestrales = [np.mean(np.random.choice(poblacion, n)) for _ in range(1000)]

print(f'\nMedia de la dist. muestral (μ_x̄):   {np.mean(medias_muestrales):.2f}  (debe ≈ μ)')
print(f'Error estándar observado (σ/√n):    {np.std(medias_muestrales):.2f}  (debe ≈ {std_poblacional/np.sqrt(n):.2f})')

In [ ]:
# Visualización de la distribución muestral
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Población original
axes[0].hist(poblacion, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(media_poblacional, color='red', linestyle='--', linewidth=2, label=f'μ = {media_poblacional:.1f}')
axes[0].set_title('Distribución de la POBLACIÓN (N=10.000)', fontsize=13)
axes[0].set_xlabel('Valor'); axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# Distribución muestral de la media
axes[1].hist(medias_muestrales, bins=30, color='skyblue', edgecolor='black', alpha=0.8)
axes[1].axvline(np.mean(medias_muestrales), color='red', linestyle='--', linewidth=2,
                label=f'μ_x̄ = {np.mean(medias_muestrales):.1f}')
axes[1].set_title('Distribución Muestral de la Media (n=30, 1000 muestras)', fontsize=13)
axes[1].set_xlabel('Media muestral'); axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 2. Ley de los Grandes Números

A medida que el tamaño de una muestra aleatoria **aumenta**, la media muestral **tiende a acercarse** a la media real de la población.

> *Cuantas más observaciones obtenemos, menor es el efecto del azar sobre los valores extremos.*

In [ ]:
# --- Código Slide 9: Ley de los Grandes Números ---

poblacion_lgn = np.random.normal(50, 10, 10000)
media_real    = np.mean(poblacion_lgn)

# Tomamos una muestra grande y calculamos medias acumulativas
muestra = np.random.choice(poblacion_lgn, 1000)
medias_acum = [np.mean(muestra[:i]) for i in range(10, 1001, 10)]
tamaños     = list(range(10, 1001, 10))

plt.figure(figsize=(12, 5))
plt.plot(tamaños, medias_acum, color='royalblue', linewidth=2, label='Media acumulativa')
plt.axhline(media_real, color='red', linestyle='--', linewidth=2,
            label=f'Media real (μ) = {media_real:.2f}')
plt.fill_between(tamaños,
                 [media_real - 2] * len(tamaños),
                 [media_real + 2] * len(tamaños),
                 alpha=0.1, color='red', label='Banda ±2 unidades')
plt.title('Ley de los Grandes Números: convergencia de la media muestral', fontsize=13)
plt.xlabel('Tamaño de muestra (n)')
plt.ylabel('Media calculada')
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()

print(f'Media real: {media_real:.2f}')
print(f'Media con n=10:   {medias_acum[0]:.2f}')
print(f'Media con n=100:  {medias_acum[9]:.2f}')
print(f'Media con n=1000: {medias_acum[-1]:.2f}')

---
## 3. Teorema del Límite Central (TLC)

Si se toman muestras suficientemente grandes de **cualquier** población con media finita y varianza finita, la distribución de las medias muestrales se **aproxima a una distribución normal**, independientemente de la forma original de los datos.

**Condiciones clave:**
- Muestras aleatorias  
- Tamaño n ≥ 30 (regla empírica)  
- Independencia entre observaciones

In [ ]:
# --- Código Slide 11: TLC con población exponencial ---

# Población NO normal (distribución exponencial)
poblacion_exp = expon.rvs(scale=2, size=10000, random_state=42)

# Distribuciones muestrales para distintos n
tamaños_n = [5, 10, 30, 100]
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, n in zip(axes, tamaños_n):
    medias = [np.mean(np.random.choice(poblacion_exp, n)) for _ in range(1000)]
    ax.hist(medias, bins=30, color='lightgreen', edgecolor='black', density=True, alpha=0.8)
    
    # Curva normal teórica
    mu_t  = np.mean(poblacion_exp)
    sig_t = np.std(poblacion_exp) / np.sqrt(n)
    x = np.linspace(min(medias), max(medias), 200)
    ax.plot(x, norm.pdf(x, mu_t, sig_t), 'r-', linewidth=2, label='Normal teórica')
    
    ax.set_title(f'n = {n}', fontsize=13)
    ax.set_xlabel('Media muestral')
    if n == tamaños_n[0]:
        ax.set_ylabel('Densidad')
    ax.legend(fontsize=8)

fig.suptitle('Teorema del Límite Central:\nDistribución exponencial → medias muestrales', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Actividad Guiada – Empresa de Logística

**Contexto:** Eres parte del equipo de ciencia de datos de una empresa de logística. La empresa no puede monitorear toda la base de datos en tiempo real, por lo que depende de análisis de muestras.

**Archivo:** `tiempos_entrega.csv`

In [ ]:
# ── PASO 1: Cargar y explorar los datos ──────────────────────────────────────

df = pd.read_csv('tiempos_entrega.csv')

print('=== Exploración inicial ===' )
print(f'Registros: {len(df)} | Columnas: {list(df.columns)}')
print()
print(df.describe().round(2))
print()
print(df.head(10))

In [ ]:
# ── Distribución original ─────────────────────────────────────────────────────

media_pob = df['tiempo_entrega_min'].mean()
std_pob   = df['tiempo_entrega_min'].std()

print(f'Media poblacional:          {media_pob:.2f} min')
print(f'Desviación estándar (σ):    {std_pob:.2f} min')
print(f'Mínimo / Máximo:            {df["tiempo_entrega_min"].min()} / {df["tiempo_entrega_min"].max()} min')

plt.figure(figsize=(10, 5))
sns.histplot(df['tiempo_entrega_min'], bins=40, kde=True, color='steelblue')
plt.axvline(media_pob, color='red', linestyle='--', linewidth=2, label=f'Media = {media_pob:.1f} min')
plt.title('Distribución original: Tiempos de entrega (min)', fontsize=13)
plt.xlabel('Tiempo de entrega (min)')
plt.ylabel('Frecuencia')
plt.legend()
plt.show()

print('\n¿La distribución parece normal? Observa la asimetría (cola a la derecha).')

In [ ]:
# ── PASO 2: Distribuciones muestrales para n = 10, 30 y 100 ─────────────────

tamaños = [10, 30, 100]
colores = ['#FF9800', '#2196F3', '#4CAF50']
poblacion_arr = df['tiempo_entrega_min'].values

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, n, color in zip(axes, tamaños, colores):
    medias = [np.mean(np.random.choice(poblacion_arr, n, replace=True)) for _ in range(500)]
    
    ax.hist(medias, bins=25, color=color, edgecolor='white', alpha=0.85, density=True)
    ax.axvline(media_pob, color='red', linestyle='--', linewidth=2, label=f'μ = {media_pob:.1f}')
    ax.axvline(np.mean(medias), color='black', linestyle='-', linewidth=2, label=f'μ_x̄ = {np.mean(medias):.1f}')
    
    # Curva normal teórica
    ee = std_pob / np.sqrt(n)
    x = np.linspace(min(medias), max(medias), 200)
    ax.plot(x, norm.pdf(x, media_pob, ee), 'k-', linewidth=1.5, linestyle=':', label='Normal teórica')
    
    ax.set_title(f'n = {n} | Error est. = {ee:.2f} min', fontsize=12)
    ax.set_xlabel('Media muestral (min)')
    if n == 10: ax.set_ylabel('Densidad')
    ax.legend(fontsize=8)

fig.suptitle('Distribuciones muestrales: Tiempos de entrega (500 muestras)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── PASO 3: Comparación numérica ─────────────────────────────────────────────

print('=== Comparación: media poblacional vs. medias muestrales ===')
print(f'{"n":>5} | {"μ_x̄ observada":>15} | {"Error estándar teórico":>22} | {"Diferencia con μ":>18}')
print('-' * 70)

for n in tamaños:
    medias = [np.mean(np.random.choice(poblacion_arr, n, replace=True)) for _ in range(500)]
    mu_obs = np.mean(medias)
    ee_teo = std_pob / np.sqrt(n)
    print(f'{n:>5} | {mu_obs:>15.2f} | {ee_teo:>22.4f} | {abs(mu_obs - media_pob):>18.4f}')

In [ ]:
# ── PASO 4: Ley de los Grandes Números con datos reales ──────────────────────

# Tomamos una muestra grande y calculamos medias acumulativas
muestra_lgn = np.random.choice(poblacion_arr, 500, replace=True)
medias_acum = [np.mean(muestra_lgn[:i]) for i in range(5, 501, 5)]
tamaños_lgn = list(range(5, 501, 5))

plt.figure(figsize=(12, 5))
plt.plot(tamaños_lgn, medias_acum, color='royalblue', linewidth=2, label='Media acumulativa')
plt.axhline(media_pob, color='red', linestyle='--', linewidth=2.5, label=f'Media poblacional = {media_pob:.2f} min')
plt.fill_between(tamaños_lgn,
                 [media_pob - 3] * len(tamaños_lgn),
                 [media_pob + 3] * len(tamaños_lgn),
                 alpha=0.1, color='red', label='Banda ±3 min')
plt.title('Ley de los Grandes Números – Tiempos de entrega', fontsize=13)
plt.xlabel('Tamaño acumulado de muestra')
plt.ylabel('Media (min)')
plt.legend()
plt.grid(True, alpha=0.4)
plt.show()

In [ ]:
# ── PASO 5: Evaluación visual del TLC ────────────────────────────────────────

from scipy.stats import shapiro

print('=== Test de normalidad de Shapiro-Wilk sobre las medias muestrales ===')
print(f'{"n":>5} | {"p-valor":>10} | {"¿Normal? (p>0.05)":>20}')
print('-' * 42)

for n in tamaños:
    medias = [np.mean(np.random.choice(poblacion_arr, n, replace=True)) for _ in range(500)]
    stat, p = shapiro(np.random.choice(medias, 200))  # Shapiro requiere n ≤ 5000
    print(f'{n:>5} | {p:>10.4f} | {"Sí ✅" if p > 0.05 else "No ❌":>20}')

---
## 5. Actividad Autónoma – Transporte Urbano

**Contexto:** Como analista en una empresa de transporte urbano, debes evaluar la confiabilidad de las estimaciones basadas en muestras del tiempo de llegada de los buses.

**Tu tarea:** Diseñar y ejecutar una simulación que responda:
*¿Con qué nivel de certeza las muestras permiten estimar correctamente la media real de los tiempos de llegada?*

---
**Completa las celdas marcadas con `# 📝 TU CÓDIGO AQUÍ`**

In [ ]:
# EJERCICIO 1 – Simula una población con distribución no normal
# Representa la variabilidad de los tiempos de llegada de buses (en minutos)
# Sugerencia: usa scipy.stats.expon u otra distribución no normal

# 📝 TU CÓDIGO AQUÍ
poblacion_buses = None  # Reemplaza con tu simulación

# Verifica tu población
# print(f'Media real: {np.mean(poblacion_buses):.2f} min')
# sns.histplot(poblacion_buses, kde=True)

In [ ]:
# EJERCICIO 2 – Extrae muestras aleatorias para n = 10, 30, 50, 100
# Para cada n, genera al menos 500 muestras y calcula la media de cada una

# 📝 TU CÓDIGO AQUÍ
tamaños_ejercicio = [10, 30, 50, 100]
resultados = {}  # Guarda aquí las medias muestrales de cada n

# for n in tamaños_ejercicio:
#     resultados[n] = [... for _ in range(500)]

In [ ]:
# EJERCICIO 3 – Grafica la distribución muestral para cada n
# Visualiza cómo cambia la forma al aumentar n (TLC)
# Superpón la curva normal teórica

# 📝 TU CÓDIGO AQUÍ

In [ ]:
# EJERCICIO 4 – Grafica la Ley de los Grandes Números
# Muestra cómo converge la media acumulativa a la media real

# 📝 TU CÓDIGO AQUÍ

In [ ]:
# EJERCICIO 5 – Tabla resumen con error estándar teórico y observado

# 📝 TU CÓDIGO AQUÍ
# Columnas sugeridas: n | μ_x̄ | Error estándar teórico | Error estándar observado

### Reflexión final

Responde las siguientes preguntas en la plantilla de resolución:

1. ¿A partir de qué tamaño de muestra la distribución de las medias se aproxima visualmente a una normal?
2. ¿Cómo varía el error estándar al duplicar el tamaño de muestra?
3. ¿Es válido aplicar modelos normales a las medias de esta población? ¿Bajo qué condiciones?
4. ¿Qué papel jugó la Ley de los Grandes Números en tus resultados?